import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import joblib


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import joblib


In [2]:
df = pd.read_csv(r"D:\ML\SM Modeling\Data\Raw\incident_enriched.csv")

print("Shape:", df.shape)
df.head(2)


Shape: (85705, 15)


,number,business_service,x_entin_itsm_app_account,short_description,priority,state,closed_at,close_code,u_resolution_sub_category,u_fault_category,close_notes,closed_by,resolved_by,incident_number,task_text
0,INC0341942,S1206097,4th Utility ISP Limited - DO NOT USE,Proactive - S1206097  TLOS 4th Utility ISP...,P5,Closed,2025-11-27 11:00:03,No Fault Found,No fault found,NaN,Our systems have seen the affected service(s) ...,NaN,NaN,NaN,NaN
1,INC0316243,S1276749,No One Internet Ltd (Trunk Networks),Proactive - S1276749  TLOS  No One Interne...,P5,Closed,2025-09-24 10:00:33,No Fault Found,No fault found,NaN,Our systems have seen the affected service(s) ...,NaN,NaN,NaN,NaN


In [3]:
df['short_description'] = df['short_description'].fillna('')
df['close_notes'] = df['close_notes'].fillna('')
df['task_text'] = df['task_text'].fillna('')

df["text"] = (
    df["short_description"] + " " +
    df["close_notes"] + " " +
    df["task_text"]
)

df = df.dropna(subset=["close_code"])
df = df.reset_index(drop=True)

print("Training rows:", len(df))


Training rows: 85705


In [4]:
label_counts = df['close_code'].value_counts()

valid_labels = label_counts[label_counts >= 25].index
df = df[df['close_code'].isin(valid_labels)].reset_index(drop=True)

print("After removing rare labels:", df.shape)
print("Unique resolution codes:", df['close_code'].nunique())


After removing rare labels: (85705, 16)
Unique resolution codes: 12


In [5]:
X = df["text"]
y = df["close_code"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # very important
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))


Train size: 68564
Test size: 17141


In [6]:
tfidf = TfidfVectorizer(
    max_features=60000,      # strong vocabulary
    ngram_range=(1,2),       # unigram + bigram
    stop_words='english',
    min_df=3,
    max_df=0.95
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("TFIDF shape:", X_train_tfidf.shape)


TFIDF shape: (68564, 60000)


In [7]:
svd = TruncatedSVD(n_components=500, random_state=42)

X_train_svd = svd.fit_transform(X_train_tfidf)
X_test_svd = svd.transform(X_test_tfidf)

print("SVD shape:", X_train_svd.shape)


SVD shape: (68564, 500)


TRAIN RESOLUTION MODEL

In [8]:
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score

model = SGDClassifier(
    loss='log_loss',          # gives probabilities
    max_iter=3000,
    n_jobs=-1,
    class_weight='balanced'
)

model.fit(X_train_svd, y_train)

pred = model.predict(X_test_svd)
acc = accuracy_score(y_test, pred)

print("\n🎯 Resolution Accuracy:", round(acc*100,2), "%")


🎯 Resolution Accuracy: 76.36 %


GET CONFIDENCE SCORES

In [9]:
import numpy as np

probs = model.predict_proba(X_test_svd)

max_probs = np.max(probs, axis=1)
pred_labels = model.predict(X_test_svd)

result_df = pd.DataFrame({
    "actual": y_test.values,
    "pred": pred_labels,
    "confidence": max_probs
})

result_df["correct"] = result_df["actual"] == result_df["pred"]

result_df.head()


,actual,pred,confidence,correct
0,Fibre Fault,Fibre Fault,0.604816,True
1,No Fault Found,Fibre Fault,0.248932,False
2,No Fault Found,No Fault Found,0.877660,True
3,No Fault Found,No Fault Found,0.938061,True
4,Fibre Fault,Fibre Fault,0.800670,True


CHECK ACCURACY BY CONFIDENCE

In [10]:
bins = [0,0.5,0.6,0.7,0.8,0.9,1.0]

result_df["conf_bucket"] = pd.cut(result_df["confidence"], bins)

report = result_df.groupby("conf_bucket").agg(
    total=("correct","count"),
    accuracy=("correct","mean")
).reset_index()

print(report)


  conf_bucket  total  accuracy
0  (0.0, 0.5]   8201  0.615413
1  (0.5, 0.6]   2776  0.849424
2  (0.6, 0.7]   2521  0.892503
3  (0.7, 0.8]   1761  0.913118
4  (0.8, 0.9]   1080  0.961111
5  (0.9, 1.0]    802  0.982544


C:\Users\anshuman.r\AppData\Local\Temp\ipykernel_21336\1468031884.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  report = result_df.groupby("conf_bucket").agg(


CALCULATE AUTO PREDICTION COVERAGE

In [11]:
auto_df = result_df[result_df["confidence"] >= 0.85]

auto_acc = auto_df["correct"].mean()
coverage = len(auto_df) / len(result_df)

print("Auto prediction accuracy:", round(auto_acc*100,2), "%")
print("Auto prediction coverage:", round(coverage*100,2), "%")


Auto prediction accuracy: 97.86 %
Auto prediction coverage: 7.36 %


    USE TOP-2 GAP CONFIDENCE (BIG BOOST)

In [12]:
# get full probability matrix
probs = model.predict_proba(X_test_svd)

top1 = np.max(probs, axis=1)
top2 = np.partition(probs, -2, axis=1)[:, -2]

gap = top1 - top2

pred_labels = model.predict(X_test_svd)

result_df = pd.DataFrame({
    "actual": y_test.values,
    "pred": pred_labels,
    "confidence": top1,
    "gap": gap
})

result_df["correct"] = result_df["actual"] == result_df["pred"]


NEW AUTO CONDITION (SMART)

In [13]:
auto_df = result_df[
    (result_df["confidence"] >= 0.75) &
    (result_df["gap"] >= 0.25)
]

auto_acc = auto_df["correct"].mean()
coverage = len(auto_df) / len(result_df)

print("Auto accuracy:", round(auto_acc*100,2), "%")
print("Coverage:", round(coverage*100,2), "%")


Auto accuracy: 95.85 %
Coverage: 15.32 %


In [14]:
pred_labels = model.predict(X_test_svd)

unique_preds = sorted(set(pred_labels))
print("Predicted Resolution Codes:\n")
for r in unique_preds:
    print(r)


Predicted Resolution Codes:

Awaiting Information
CityFibre Network
Customer
Equipment Fault
Fibre Fault
Internal
Maintenance
No Fault Found
Offnet Supplier
Power Failure
Service Disturbance
Service Request


In [15]:
import pandas as pd

pred_labels = model.predict(X_test_svd)

pred_series = pd.Series(pred_labels)
print(pred_series.value_counts())


Fibre Fault             6702
No Fault Found          4061
Customer                1836
Equipment Fault         1716
Internal                 700
Service Request          626
Awaiting Information     556
Service Disturbance      432
Maintenance              316
CityFibre Network        112
Power Failure             66
Offnet Supplier           18
Name: count, dtype: int64


In [16]:
actual_counts = pd.Series(y_test).value_counts()
pred_counts = pd.Series(pred_labels).value_counts()

compare_df = pd.DataFrame({
    "actual": actual_counts,
    "predicted": pred_counts
}).fillna(0)

print(compare_df)


                      actual  predicted
Awaiting Information     406        556
CityFibre Network          9        112
Customer                2091       1836
Equipment Fault         1696       1716
Fibre Fault             6580       6702
Internal                 530        700
Maintenance              165        316
No Fault Found          4849       4061
Offnet Supplier           20         18
Power Failure             53         66
Service Disturbance      313        432
Service Request          429        626


In [17]:
probs = model.predict_proba(X_test_svd)
confidence = probs.max(axis=1)

pred_df = pd.DataFrame({
    "text": X_test.values,
    "actual": y_test.values,
    "predicted": pred_labels,
    "confidence": confidence
})

pred_df.head(20)


,text,actual,predicted,confidence
0,S44149 Service Outage Damage has been exposed ...,Fibre Fault,Fibre Fault,0.604816
1,S1176472-FTTH-Line test failed TLOS Please can...,No Fault Found,Fibre Fault,0.248932
2,S377514-FTTH-Walking in router not in syncChec...,No Fault Found,No Fault Found,0.877660
3,S313172 Service Outage Our systems have seen t...,No Fault Found,No Fault Found,0.938061
4,S385243-FTTH-Service outage What was the fault...,Fibre Fault,Fibre Fault,0.800670
5,Proactive - S1435587  TLOS  Vodafone Limited...,No Fault Found,No Fault Found,0.827605
6,S1182619-Business FTTP-Line test failed LOS Ou...,No Fault Found,No Fault Found,0.884079
7,S1141940-Broadband-Ethernet light is not flash...,Fibre Fault,Fibre Fault,0.676923
8,S1699963-FTTH-the broadband light keeps flashi...,Fibre Fault,Equipment Fault,0.531216
9,S1286473 - Slow speeds ETA 22/7 AM Everything ...,Customer,Customer,0.250845


In [18]:
import joblib, os
save_dir = r"D:\ML\SM Modeling\AI_ENGINE_V3\models"
os.makedirs(save_dir, exist_ok=True)

joblib.dump(model, save_dir + r"\resolution_model.pkl")
joblib.dump(tfidf, save_dir + r"\res_tfidf.pkl")
joblib.dump(svd, save_dir + r"\res_svd.pkl")

print("Resolution model saved")


Resolution model saved
